# EAF Heat Temperature Prediction

**Project:** Prediction of final EAF bath temperature from process parameters

The idea is simple: temperature is measured during an electric arc furnace heat, but the useful question is whether the process data available before the final measurement can give us a good estimate of the final temperature.

I am using the public **Industrial Data from the Electric Arc Furnace** dataset from Kaggle.

### Main steps
- download and inspect the raw EAF files
- create one row per heat
- aggregate process information available before the final temperature reading
- train a few regression models
- compare the results
- look at which process variables have the most influence

**Important:** final chemical measurements are not used as predictors because they are available after/around the end of the process and would create leakage.


## 1. Install / import libraries

In [ ]:
# If you are running this in Google Colab, run this cell first.
!pip -q install kagglehub scikit-learn joblib seaborn

import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

pd.set_option("display.max_columns", 100)


## 2. Download the Kaggle dataset

The raw dataset is quite large, so I would **not upload all of it to GitHub**.

The notebook downloads it into:

```text
data/raw/
```

If KaggleHub does not work in your environment, download the dataset manually from Kaggle and extract all CSV files into `data/raw/`.

Dataset:
**Industrial Data from the Electric Arc Furnace**


In [ ]:
import kagglehub
import shutil
from pathlib import Path

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

dataset_path = kagglehub.dataset_download(
    "yuriykatser/industrial-data-from-the-arc-furnace"
)

print("Downloaded to:", dataset_path)

# Copy CSV files into our project data/raw folder.
csv_files = list(Path(dataset_path).rglob("*.csv"))

for f in csv_files:
    shutil.copy2(f, DATA_DIR / f.name)

print("CSV files copied:", len(csv_files))
print(sorted([f.name for f in DATA_DIR.glob("*.csv")]))


## 3. Check the files

In [ ]:
files = sorted(DATA_DIR.glob("*.csv"))

for f in files:
    size_mb = f.stat().st_size / (1024**2)
    print(f"{f.name:45s} {size_mb:8.1f} MB")


In [ ]:
# Load the temperature table first.
temp = pd.read_csv(DATA_DIR / "eaf_temp.csv")

print(temp.shape)
display(temp.head())
print(temp.dtypes)


## 4. Temperature data

`eaf_temp.csv` contains:
- `HEATID` - heat identification
- `DATETIME` - measurement time
- `TEMP` - measured temperature
- `VALO2_PPM` - oxidation degree

For this project, the **last valid temperature measurement of each heat** is treated as the target. Earlier temperature measurements are useful predictors because they are known before the final reading.


In [ ]:
temp["DATETIME"] = pd.to_datetime(temp["DATETIME"], errors="coerce")
temp["TEMP"] = pd.to_numeric(temp["TEMP"], errors="coerce")

temp = temp.dropna(subset=["HEATID", "DATETIME", "TEMP"]).copy()
temp = temp.sort_values(["HEATID", "DATETIME"])

target = (
    temp.groupby("HEATID", as_index=False)
        .tail(1)
        [["HEATID", "DATETIME", "TEMP", "VALO2_PPM"]]
        .rename(columns={
            "DATETIME": "TARGET_TIME",
            "TEMP": "TARGET_TEMP",
            "VALO2_PPM": "TARGET_OXIDATION"
        })
)

print(target.shape)
display(target.head())


## 5. Create heat-level temperature features

For every heat, I keep information from the temperature measurements **before the final measurement**.

This avoids using the target itself as an input.


In [ ]:
before = temp.merge(
    target[["HEATID", "TARGET_TIME"]],
    on="HEATID",
    how="inner"
)

before = before[before["DATETIME"] < before["TARGET_TIME"]].copy()

temp_features = (
    before.groupby("HEATID")
    .agg(
        INITIAL_TEMP=("TEMP", "first"),
        PREV_TEMP=("TEMP", "last"),
        MEAN_TEMP=("TEMP", "mean"),
        MIN_TEMP=("TEMP", "min"),
        MAX_PREV_TEMP=("TEMP", "max"),
        TEMP_MEASUREMENTS=("TEMP", "count"),
        MEAN_OXIDATION=("VALO2_PPM", "mean"),
        MAX_OXIDATION=("VALO2_PPM", "max"),
    )
    .reset_index()
)

display(temp_features.head())


## 6. Transformer / electrical features

The transformer table records furnace electrical operation in stages. I aggregate only the records whose start time is before the target temperature measurement.


In [ ]:
tr = pd.read_csv(DATA_DIR / "eaf_transformer.csv")

tr["STARTTIME"] = pd.to_datetime(tr["STARTTIME"], errors="coerce")
tr["MW"] = pd.to_numeric(tr["MW"], errors="coerce")
tr["DURATION"] = pd.to_numeric(tr["DURATION"], errors="coerce")
tr["TAP"] = pd.to_numeric(tr["TAP"], errors="coerce")

tr = tr.dropna(subset=["HEATID", "STARTTIME"])

tr = tr.merge(target[["HEATID", "TARGET_TIME"]], on="HEATID", how="inner")
tr = tr[tr["STARTTIME"] <= tr["TARGET_TIME"]].copy()

tr_features = (
    tr.groupby("HEATID")
    .agg(
        TRANSFORMER_OPERATIONS=("HEATID", "size"),
        TOTAL_MW=("MW", "sum"),
        MEAN_MW=("MW", "mean"),
        MAX_MW=("MW", "max"),
        TOTAL_DURATION=("DURATION", "sum"),
        MEAN_DURATION=("DURATION", "mean"),
        MEAN_TAP=("TAP", "mean"),
        MAX_TAP=("TAP", "max"),
    )
    .reset_index()
)

display(tr_features.head())


## 7. Gas and oxygen features

The gas/oxygen table contains cumulative process quantities and flow rates. For cumulative fields, the maximum value reached before the target measurement is more meaningful than simply summing every row.


In [ ]:
gas = pd.read_csv(DATA_DIR / "eaf_gaslance_mat.csv")

gas["REVTIME"] = pd.to_datetime(gas["REVTIME"], errors="coerce")

numeric_gas = [
    "O2_AMOUNT", "GAS_AMOUNT", "O2_FLOW", "GAS_FLOW"
]
for c in numeric_gas:
    gas[c] = pd.to_numeric(gas[c], errors="coerce")

gas = gas.dropna(subset=["HEATID", "REVTIME"])
gas = gas.merge(target[["HEATID", "TARGET_TIME"]], on="HEATID", how="inner")
gas = gas[gas["REVTIME"] <= gas["TARGET_TIME"]].copy()

gas_features = (
    gas.groupby("HEATID")
    .agg(
        O2_USED=("O2_AMOUNT", "max"),
        GAS_USED=("GAS_AMOUNT", "max"),
        MEAN_O2_FLOW=("O2_FLOW", "mean"),
        MAX_O2_FLOW=("O2_FLOW", "max"),
        MEAN_GAS_FLOW=("GAS_FLOW", "mean"),
        MAX_GAS_FLOW=("GAS_FLOW", "max"),
    )
    .reset_index()
)

display(gas_features.head())


## 8. Carbon injection features

In [ ]:
inj = pd.read_csv(DATA_DIR / "inj_mat.csv")
inj["REVTIME"] = pd.to_datetime(inj["REVTIME"], errors="coerce")

for c in ["INJ_AMOUNT_CARBON", "INJ_FLOW_CARBON"]:
    inj[c] = pd.to_numeric(inj[c], errors="coerce")

inj = inj.dropna(subset=["HEATID", "REVTIME"])
inj = inj.merge(target[["HEATID", "TARGET_TIME"]], on="HEATID", how="inner")
inj = inj[inj["REVTIME"] <= inj["TARGET_TIME"]].copy()

inj_features = (
    inj.groupby("HEATID")
    .agg(
        CARBON_USED=("INJ_AMOUNT_CARBON", "max"),
        MEAN_CARBON_FLOW=("INJ_FLOW_CARBON", "mean"),
        MAX_CARBON_FLOW=("INJ_FLOW_CARBON", "max"),
    )
    .reset_index()
)

display(inj_features.head())


## 9. Charge/addition features

Instead of creating thousands of one-hot columns for material names, I first create general process features such as total charged mass and number of additions.

This keeps the first model reasonably interpretable.


In [ ]:
def addition_features(filename, time_col, amount_col):
    df = pd.read_csv(DATA_DIR / filename)
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df[amount_col] = pd.to_numeric(df[amount_col], errors="coerce")

    df = df.dropna(subset=["HEATID", time_col])
    df = df.merge(target[["HEATID", "TARGET_TIME"]], on="HEATID", how="inner")
    df = df[df[time_col] <= df["TARGET_TIME"]].copy()

    out = (
        df.groupby("HEATID")
        .agg(
            TOTAL_ADDED=(amount_col, "sum"),
            MAX_ADDED=(amount_col, "max"),
            NUMBER_OF_ADDITIONS=(amount_col, "size"),
        )
        .reset_index()
    )
    return out

basket_features = addition_features(
    "basket_charged.csv", "DATETIME", "CHARGE_AMOUNT"
).rename(columns={
    "TOTAL_ADDED": "BASKET_TOTAL",
    "MAX_ADDED": "BASKET_MAX",
    "NUMBER_OF_ADDITIONS": "BASKET_ADDITIONS"
})

extra_features = addition_features(
    "eaf_added_materials.csv", "DATETIME", "CHARGE_AMOUNT"
).rename(columns={
    "TOTAL_ADDED": "EXTRA_TOTAL",
    "MAX_ADDED": "EXTRA_MAX",
    "NUMBER_OF_ADDITIONS": "EXTRA_ADDITIONS"
})

display(basket_features.head())
display(extra_features.head())


## 10. Merge everything into one modelling table

In [ ]:
df = target[["HEATID", "TARGET_TIME", "TARGET_TEMP"]].copy()

tables = [
    temp_features,
    tr_features,
    gas_features,
    inj_features,
    basket_features,
    extra_features,
]

for t in tables:
    df = df.merge(t, on="HEATID", how="left")

# Time-related process feature
df["PROCESS_DURATION_MIN"] = (
    df["TARGET_TIME"] -
    temp.groupby("HEATID")["DATETIME"].min().rename("FIRST_MEASUREMENT")
        .reindex(df["HEATID"]).values
).dt.total_seconds() / 60

df = df.replace([np.inf, -np.inf], np.nan)

print("Final shape:", df.shape)
display(df.head())


## 11. Basic data cleaning

In [ ]:
# Keep numeric predictors only.
drop_cols = ["HEATID", "TARGET_TIME", "TARGET_TEMP"]
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df["TARGET_TEMP"].copy()

# Remove columns that are completely empty.
X = X.dropna(axis=1, how="all")

print("Features:", list(X.columns))
print("Missing values:")
display(X.isna().sum().sort_values(ascending=False).head(15))


## 12. Train/test split

A random split is convenient, but for an industrial process a time-based split is more realistic.

The earlier heats are used for training and the later heats for testing. This better represents the situation where a model is trained on historical production data and then used on future heats.


In [ ]:
model_df = pd.concat([df[["TARGET_TIME", "TARGET_TEMP"]], X], axis=1)
model_df = model_df.sort_values("TARGET_TIME").reset_index(drop=True)

split = int(len(model_df) * 0.80)

train_df = model_df.iloc[:split].copy()
test_df = model_df.iloc[split:].copy()

X_train = train_df.drop(columns=["TARGET_TIME", "TARGET_TEMP"])
y_train = train_df["TARGET_TEMP"]

X_test = test_df.drop(columns=["TARGET_TIME", "TARGET_TEMP"])
y_test = test_df["TARGET_TEMP"]

print("Train:", X_train.shape)
print("Test :", X_test.shape)


## 13. Train a few regression models

In [ ]:
models = {
    "Ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=250,
            max_depth=12,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "HistGradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            random_state=42
        ))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)


## 14. Actual vs predicted temperature

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = models[best_name]
best_pred = best_model.predict(X_test)

plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_pred, alpha=0.45)
lims = [
    min(y_test.min(), best_pred.min()),
    max(y_test.max(), best_pred.max())
]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual temperature (°C)")
plt.ylabel("Predicted temperature (°C)")
plt.title(f"Actual vs Predicted - {best_name}")
plt.show()


## 15. Prediction error

In [ ]:
errors = y_test - best_pred

plt.figure(figsize=(8, 4))
plt.hist(errors, bins=35)
plt.xlabel("Actual - Predicted (°C)")
plt.ylabel("Number of heats")
plt.title("Prediction error distribution")
plt.show()

print("Mean error:", errors.mean())
print("Mean absolute error:", np.abs(errors).mean())


## 16. Feature importance

Permutation importance is used here because it works with the final fitted pipeline and gives a simple view of which process variables are useful for prediction.


In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=8,
    random_state=42,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean
}).sort_values("Importance", ascending=False)

display(importance.head(15))


In [ ]:
top = importance.head(12).sort_values("Importance")

plt.figure(figsize=(8, 5))
plt.barh(top["Feature"], top["Importance"])
plt.xlabel("Permutation importance")
plt.title("Top process variables")
plt.show()


## 17. Save the model

The model can later be loaded for a new heat once the same feature-engineering process is applied.


In [ ]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    {
        "model": best_model,
        "features": list(X_train.columns),
        "target": "TARGET_TEMP"
    },
    MODEL_DIR / "eaf_temperature_model.joblib"
)

print("Saved:", MODEL_DIR / "eaf_temperature_model.joblib")


## 18. A small prediction example

This is only an example using one row from the test data. In an actual plant system, the row would come from the current heat's process data.


In [ ]:
sample = X_test.iloc[[0]]
sample_prediction = best_model.predict(sample)[0]

print(f"Predicted final EAF temperature: {sample_prediction:.1f} °C")
print(f"Actual final EAF temperature:    {y_test.iloc[0]:.1f} °C")
print(f"Absolute error:                 {abs(sample_prediction-y_test.iloc[0]):.1f} °C")


## 19. Conclusion

The model is intended as a **process-monitoring / prediction prototype**, not as a direct furnace-control system.

The main learning from this project is the workflow:

**raw industrial event data → time alignment → heat-level feature engineering → leakage control → regression → evaluation → process interpretation**

Possible next improvements:
- predict oxidation degree as a second target
- add material-category features from `MAT_DEC`
- tune the best model with cross-validation
- compare XGBoost/CatBoost
- build a Power BI dashboard around heat-level predictions
- create an operating-window analysis using predicted temperature error
